# Part 3 — Do Humans Even Agree? Annotator Agreement by Hand

*Evals from First Principles, Part 3.*

Your labels have noise; measure it before you trust them. Raw percent-agreement lies, because two raters agree by chance whenever the labels are imbalanced. This notebook builds Cohen's kappa by hand from a 2x2 (the chance-correction that collapses a scary-high 0.90 raw agreement to a modest number), then Fleiss' kappa for three raters, and closes with a one-line note on Krippendorff's alpha as the general case.

Pure Python, no imports, fully offline and deterministic — every number below is reproducible on a laptop with no network and no API key.

Companion script: `agreement.py`

## Step 1 — Two raters, 20 tickets

Two human annotators grade the SAME 20 support tickets: "urgent" (1) or "not urgent" (0). `RATER_A[i]` and `RATER_B[i]` describe the same ticket `i`. Most tickets are calm, so both raters say "not urgent" most of the time — which is exactly what will inflate raw agreement.

In [ ]:
RATER_A = [1, 1, 1, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
RATER_B = [1, 1, 0, 1,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

print("TWO RATERS, 20 TICKETS: each labels 'urgent' (1) or 'not urgent' (0).")
print()
print("  item  A  B  agree")
for i, (a, b) in enumerate(zip(RATER_A, RATER_B), start=1):
    print(f"  {i:>4}  {a}  {b}  {'yes' if a == b else 'NO '}")

## Step 2 — The 2x2, counted by hand

Every pair of labels on the same ticket lands in exactly one of four buckets: both said urgent, A said urgent but B did not, A did not but B said urgent, or both said not urgent. Raw agreement `po` is just `(both-urgent + both-not) / n` — nothing more than the fraction of tickets where the two raters landed in the same bucket.

In [ ]:
def cohen_cells(a_labels, b_labels):
    """Fill the 2x2 of two raters by hand: (both1, a1b0, a0b1, both0)."""
    both1 = a1b0 = a0b1 = both0 = 0
    for a, b in zip(a_labels, b_labels):
        if a == 1 and b == 1:
            both1 += 1
        elif a == 1 and b == 0:
            a1b0 += 1
        elif a == 0 and b == 1:
            a0b1 += 1
        else:  # a == 0 and b == 0
            both0 += 1
    return both1, a1b0, a0b1, both0


both1, a1b0, a0b1, both0 = cohen_cells(RATER_A, RATER_B)
n = len(RATER_A)
print("  2x2 of A vs B (counted by hand):")
print(f"                 B=urgent   B=not")
print(f"    A=urgent  {both1:>7}   {a1b0:>5}")
print(f"    A=not     {a0b1:>7}   {both0:>5}")
print()
po = (both1 + both0) / n
print(f"  raw agreement po = (both-urgent + both-not)/n = ({both1}+{both0})/{n} = {po:.2f}")
print(f"  ...a scary-high {po:.0%}. But how much of that is just chance?")

## Step 3 — Chance-corrected agreement: Cohen's kappa

A raw 90% looks great, until you notice both raters say "not urgent" almost every time. Two coin flips that are both biased toward "not urgent" will *also* agree most of the time, purely by chance. `pe` estimates exactly that: how often the two raters would agree if they were guessing independently, using only their own marginal rates. Cohen's kappa then rescales the gap between observed and chance agreement onto a 0-to-1-ish scale: `(po - pe) / (1 - pe)`.

In [ ]:
def cohen_kappa(both1, a1b0, a0b1, both0):
    """(po - pe) / (1 - pe): observed agreement corrected for chance."""
    n = both1 + a1b0 + a0b1 + both0
    po = (both1 + both0) / n
    # marginal chance each rater says "urgent" / "not urgent"
    a_urg = (both1 + a1b0) / n
    b_urg = (both1 + a0b1) / n
    a_not, b_not = 1 - a_urg, 1 - b_urg
    pe = a_urg * b_urg + a_not * b_not
    kappa = (po - pe) / (1 - pe)
    return po, pe, kappa, (a_urg, b_urg, a_not, b_not)


po, pe, kappa, (a_urg, b_urg, a_not, b_not) = cohen_kappa(both1, a1b0, a0b1, both0)
print(f"  each rater's marginal rates:")
print(f"    A says urgent {both1 + a1b0}/{n} = {a_urg:.2f},  not {a_not:.2f}")
print(f"    B says urgent {both1 + a0b1}/{n} = {b_urg:.2f},  not {b_not:.2f}")
print(f"  chance agreement pe = {a_urg:.2f}*{b_urg:.2f} + {a_not:.2f}*{b_not:.2f}")
print(f"                      = {a_urg * b_urg:.4f} + {a_not * b_not:.4f} = {pe:.4f}")
print()
print(f"  Cohen kappa = (po - pe)/(1 - pe)")
print(f"              = ({po:.2f} - {pe:.4f})/(1 - {pe:.4f}) = {kappa:.2f}")
print(f"  -> {po:.0%} raw agreement is really kappa {kappa:.2f}: two raters who")
print(f"     almost always say 'not urgent' agree by chance {pe:.0%} of the time,")
print(f"     so the honest, chance-corrected number is far lower.")

## Step 4 — Three raters: Fleiss' kappa

Cohen's kappa only handles two raters. The moment a third rater joins, we need a different formula: Fleiss' kappa. Five fresh tickets, each graded by three raters R1, R2, R3 (1 = urgent, 0 = not). For each ticket we only need the *counts* per category, not who said what — Fleiss' kappa works on vote tallies.

In [ ]:
R1 = [1, 1, 0, 1, 0]
R2 = [1, 1, 0, 0, 0]
R3 = [1, 0, 0, 0, 0]
N_RATERS = 3

print("THREE RATERS: Cohen only handles two, so use Fleiss' kappa.")
print("Five fresh tickets, each graded by raters R1, R2, R3 (1=urgent, 0=not).")
print()
print("  item  R1 R2 R3  | urgent  not")
counts = []
for i in range(len(R1)):
    votes = [R1[i], R2[i], R3[i]]
    n_urg = sum(votes)
    n_not = N_RATERS - n_urg
    counts.append([n_not, n_urg])  # [category 0 = not, category 1 = urgent]
    print(f"  {i + 1:>4}   {R1[i]}  {R2[i]}  {R3[i]}  |{n_urg:>6}  {n_not:>4}")

## Step 5 — Fleiss' kappa by hand

For each item, `P_i` is the fraction of rater *pairs* that agree: take the sum of squared per-category vote counts, subtract the number of raters, and divide by `raters * (raters - 1)`. Average `P_i` across items to get `P_bar`, the observed agreement. Chance agreement `P_e` uses the overall share of votes landing in each category across the whole table, squared and summed — the same chance-correction idea as Cohen's kappa, just generalized to any number of raters.

In [ ]:
def fleiss_kappa(counts, n_raters):
    """Fleiss' kappa from a per-item category-count table (any # of raters).

    counts[i] = [n_cat0, n_cat1] = how many raters put item i in each category.
    Returns (P_bar, P_e, kappa, item_agreements, category_props).
    """
    n_items = len(counts)
    # P_i: fraction of rater PAIRS on item i that agree
    item_agree = []
    for row in counts:
        s = sum(c * c for c in row)
        item_agree.append((s - n_raters) / (n_raters * (n_raters - 1)))
    p_bar = sum(item_agree) / n_items
    # p_j: overall share of assignments landing in each category
    total = n_items * n_raters
    cat_props = [sum(row[j] for row in counts) / total for j in range(len(counts[0]))]
    p_e = sum(p * p for p in cat_props)
    kappa = (p_bar - p_e) / (1 - p_e)
    return p_bar, p_e, kappa, item_agree, cat_props


p_bar, p_e, fk, item_agree, cat_props = fleiss_kappa(counts, N_RATERS)
print("  per-item agreement P_i = (sum of squared vote-counts - raters)")
print("                           / (raters * (raters - 1)):")
for i, pi in enumerate(item_agree, start=1):
    print(f"    item {i}: P_i = {pi:.4f}")
print(f"  mean observed agreement  P_bar = {p_bar:.4f}")
p_not, p_urg = cat_props
print(f"  category shares: urgent {p_urg:.2f}, not {p_not:.2f}")
print(f"  chance agreement P_e = {p_urg:.2f}^2 + {p_not:.2f}^2 = {p_e:.2f}")
print(f"  Fleiss kappa = (P_bar - P_e)/(1 - P_e)")
print(f"               = ({p_bar:.4f} - {p_e:.2f})/(1 - {p_e:.2f}) = {fk:.2f}")

## Recap

- Raw percent-agreement is misleading whenever labels are imbalanced: two raters who mostly say "not urgent" agree with each other by chance most of the time, inflating `po` without telling you anything about real agreement.
- **Cohen's kappa** `(po - pe) / (1 - pe)` corrects for exactly that: it collapsed our scary-high 90% raw agreement down to a modest kappa of 0.61.
- **Fleiss' kappa** generalizes the same chance-correction to three or more raters, working from per-item vote counts instead of a 2x2: our three-rater set landed at kappa 0.44.
- **Krippendorff's alpha** is the general case beyond both: any number of raters, missing labels, and any scale (nominal, ordinal, interval). It is built from disagreement instead of agreement, but the underlying idea is the same one we just did by hand: correct the raw number for chance before you trust it.

Next: what happens once your gold labels come not from humans, but from another LLM acting as judge.